# Data Validation

This notebook validates the stacked TIFF dataset in `data/stacked_raw` and provides a quick visual and statistical sanity check.

Checks included:
- file discovery and path setup
- full-dataset shape / dtype / readability validation
- sample-level descriptive statistics
- channel-level inspection for one example file
- compatibility check with `src/dataset.py`


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile
from IPython.display import display

plt.style.use("ggplot")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "stacked_raw"
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Source directory: {SRC_DIR.resolve()}")


In [ ]:
tif_paths = sorted(DATA_DIR.glob("*.tif"))
assert tif_paths, f"No TIFF files found under {DATA_DIR}"

print(f"Found {len(tif_paths):,} TIFF files.")
display(pd.DataFrame({"example_file": [p.name for p in tif_paths[:5]]}))


In [ ]:
records = []
errors = []

for path in tif_paths:
    try:
        arr = tifffile.imread(path)
        arr = np.asarray(arr)
        records.append(
            {
                "file": path.name,
                "shape": tuple(arr.shape),
                "dtype": str(arr.dtype),
                "ndim": int(arr.ndim),
                "channels": int(arr.shape[0]) if arr.ndim == 3 else np.nan,
                "height": int(arr.shape[-2]) if arr.ndim >= 2 else np.nan,
                "width": int(arr.shape[-1]) if arr.ndim >= 2 else np.nan,
                "min": float(arr.min()),
                "max": float(arr.max()),
                "mean": float(arr.mean()),
                "std": float(arr.std()),
            }
        )
    except Exception as exc:
        errors.append({"file": path.name, "error": repr(exc)})

validation_df = pd.DataFrame.from_records(records)
errors_df = pd.DataFrame.from_records(errors)

print(f"Readable files: {len(validation_df):,}")
print(f"Read errors: {len(errors_df):,}")

if not errors_df.empty:
    display(errors_df.head())

display(validation_df.head())


In [ ]:
expected_shape = (59, 100, 100)
expected_dtype = "uint16"

summary = pd.Series(
    {
        "num_files": len(tif_paths),
        "num_readable": len(validation_df),
        "num_errors": len(errors_df),
        "unique_shapes": validation_df["shape"].nunique(),
        "unique_dtypes": validation_df["dtype"].nunique(),
        "all_shape_match_expected": bool((validation_df["shape"] == expected_shape).all()),
        "all_dtype_match_expected": bool((validation_df["dtype"] == expected_dtype).all()),
        "all_channel_count_59": bool((validation_df["channels"] == 59).all()),
    }
).to_frame(name="value")

display(summary)

shape_counts = validation_df["shape"].value_counts().rename_axis("shape").reset_index(name="count")
dtype_counts = validation_df["dtype"].value_counts().rename_axis("dtype").reset_index(name="count")

display(shape_counts)
display(dtype_counts)

assert errors_df.empty, "Some TIFF files could not be read."
assert (validation_df["shape"] == expected_shape).all(), "Found unexpected shapes."
assert (validation_df["dtype"] == expected_dtype).all(), "Found unexpected dtypes."


In [ ]:
sample_stats = validation_df[["file", "min", "max", "mean", "std"]].copy()
display(sample_stats.describe().T)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(sample_stats["max"], bins=40, color="#4C78A8", edgecolor="white")
axes[0].set_title("Distribution of per-file max")

axes[1].hist(sample_stats["mean"], bins=40, color="#F58518", edgecolor="white")
axes[1].set_title("Distribution of per-file mean")

axes[2].hist(sample_stats["std"], bins=40, color="#54A24B", edgecolor="white")
axes[2].set_title("Distribution of per-file std")

plt.tight_layout()
plt.show()


In [ ]:
sample_idx = 0
sample_path = tif_paths[sample_idx]
sample_stack = tifffile.imread(sample_path).astype(np.float32)

channel_stats = pd.DataFrame(
    {
        "channel": np.arange(sample_stack.shape[0]),
        "min": sample_stack.min(axis=(1, 2)),
        "max": sample_stack.max(axis=(1, 2)),
        "mean": sample_stack.mean(axis=(1, 2)),
        "std": sample_stack.std(axis=(1, 2)),
    }
)

print(f"Inspecting: {sample_path.name}")
display(channel_stats.head(10))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(channel_stats["channel"], channel_stats["mean"], marker="o", color="#E45756")
axes[0].set_title("Per-channel mean intensity")
axes[0].set_xlabel("channel")
axes[0].set_ylabel("mean")

axes[1].plot(channel_stats["channel"], channel_stats["std"], marker="o", color="#72B7B2")
axes[1].set_title("Per-channel std")
axes[1].set_xlabel("channel")
axes[1].set_ylabel("std")

plt.tight_layout()
plt.show()


In [ ]:
from dataset import MicroscopyTiffDataset, get_default_input_indices

target_index = 10
dataset = MicroscopyTiffDataset(tif_paths=tif_paths[:8], target_index=target_index)
x, y = dataset[0]

print("Default input indices:", get_default_input_indices())
print("x shape:", tuple(x.shape), x.dtype)
print("y shape:", tuple(y.shape), y.dtype)

for i in range(x.shape[0]):
    ch = x[i]
    print(
        f"x[{i}] -> min={ch.min().item():.4f}, max={ch.max().item():.4f}, mean={ch.mean().item():.4f}"
    )

print(
    f"y[0] -> min={y[0].min().item():.4f}, max={y[0].max().item():.4f}, mean={y[0].mean().item():.4f}"
)


In [ ]:
fig, axes = plt.subplots(1, 7, figsize=(18, 3))

for i in range(6):
    axes[i].imshow(x[i].cpu().numpy(), cmap="gray")
    axes[i].set_title(f"x[{i}]")
    axes[i].axis("off")

axes[6].imshow(y[0].cpu().numpy(), cmap="gray")
axes[6].set_title(f"target[{target_index}]")
axes[6].axis("off")

plt.tight_layout()
plt.show()


## Result

If all assertions above pass, the raw dataset is internally consistent for the expected `(59, 100, 100)` stacked TIFF format and the dataset loader can read a sample end-to-end.
